# Data Preparation for NFT Market Regimes Dataset

This notebook reproduces the public-data ingestion and preparation steps used to build the BigQuery tables required for the analysis-ready transaction dataset.


## 0. Before you start

This notebook assumes:

- the recommended execution environment defined in `environment.yml`,
- a VS Code Notebook workflow,
- a configured GCP project, BigQuery dataset, and GCS bucket,
- local authentication via `gcloud auth login` and `gcloud auth application-default login`.

The notebook covers:

- downloading or placing the public source files locally,
- uploading local files to GCS,
- loading `nft_trading_base`, `nft_metadata_base`, and `usd_eth_raw`,
- normalizing `usd_eth_raw` into `usd_eth_base`,
- creating `nft_trading_usd`,
- creating `nft_trading_usd_prefilter`,
- creating `nft_trading_usd_filtered`.

Configuration values for project, bucket, analysis period, and filtering thresholds are defined in the first code cell and reused throughout the notebook.


In [ ]:
from pathlib import Path
import pandas as pd
from google.cloud import bigquery, storage

# --- Environment-specific identifiers ---
PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
DATASET_ID = "<YOUR_BIGQUERY_DATASET>"
LOCATION = "asia-northeast1"
BUCKET_NAME = "<YOUR_GCS_BUCKET>"
GCS_RAW_PREFIX = "nft_market_regimes/raw"

# --- Local paths ---
LOCAL_RAW_DIR = Path("data/raw")
LOCAL_EXTRACTED_DIR = Path("data/extracted")

# --- Reusable analysis parameters ---
ANALYSIS_START_DATE = "2017-10-19"
ANALYSIS_END_DATE = "2025-04-01"
GLOBAL_MAX_PRICE_USD = 5_000_000
MIN_ACTIVE_WEEKS = 8
LOCAL_PRICE_QUANTILE_RESOLUTION = 1000
LOCAL_PRICE_QUANTILE_OFFSET = 999

BQ_DATASET = f"{PROJECT_ID}.{DATASET_ID}"
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(project=PROJECT_ID)

print("PROJECT_ID:", PROJECT_ID)
print("BQ_DATASET:", BQ_DATASET)
print("BUCKET_NAME:", BUCKET_NAME)
print("ANALYSIS_START_DATE:", ANALYSIS_START_DATE)
print("ANALYSIS_END_DATE:", ANALYSIS_END_DATE)


## 1. Download public source files

Place the following files locally before running the rest of the notebook:

- `data/raw/nft_trading_base.tar.gz`
- `data/raw/nft_metadata_base.parquet.gz`
- `data/raw/etherprice.csv`

Then extract the trading archive so that Parquet parts are available under `data/extracted/nft_trading_base/`.


In [ ]:
# Inspect local files
print("raw dir exists:", LOCAL_RAW_DIR.exists())
print("extracted dir exists:", LOCAL_EXTRACTED_DIR.exists())
print(sorted([p.name for p in LOCAL_RAW_DIR.glob('*')]))


## 2. Upload source files to GCS

You can upload with either CLI commands or Python.

### CLI examples

```bash
gcloud storage cp data/raw/nft_metadata_base.parquet.gz gs://<YOUR_GCS_BUCKET>/nft_market_regimes/raw/
gcloud storage cp data/raw/etherprice.csv gs://<YOUR_GCS_BUCKET>/nft_market_regimes/raw/
gcloud storage cp --recursive data/extracted/nft_trading_base gs://<YOUR_GCS_BUCKET>/nft_market_regimes/raw/nft_trading_base/
```


In [ ]:
def upload_file_to_gcs(local_path: Path, bucket_name: str, blob_name: str) -> None:
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(str(local_path))
    print(f"Uploaded: {local_path} -> gs://{bucket_name}/{blob_name}")


def upload_directory_parquet_to_gcs(local_dir: Path, bucket_name: str, prefix: str) -> None:
    for path in sorted(local_dir.rglob("*.parquet")):
        rel = path.relative_to(local_dir)
        blob_name = f"{prefix}/{rel.as_posix()}"
        upload_file_to_gcs(path, bucket_name, blob_name)

# Example usage:
# upload_file_to_gcs(LOCAL_RAW_DIR / "nft_metadata_base.parquet.gz", BUCKET_NAME, f"{GCS_RAW_PREFIX}/nft_metadata_base.parquet.gz")
# upload_file_to_gcs(LOCAL_RAW_DIR / "etherprice.csv", BUCKET_NAME, f"{GCS_RAW_PREFIX}/etherprice.csv")
# upload_directory_parquet_to_gcs(LOCAL_EXTRACTED_DIR / "nft_trading_base", BUCKET_NAME, f"{GCS_RAW_PREFIX}/nft_trading_base")


## 3. Create dataset if needed


In [ ]:
dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET_ID}")


## 4. Load `nft_metadata_base` from parquet

CLI equivalent:

```bash
bq load   --source_format=PARQUET   --replace   <YOUR_GCP_PROJECT_ID>:<YOUR_BIGQUERY_DATASET>.nft_metadata_base   gs://<YOUR_GCS_BUCKET>/nft_market_regimes/raw/nft_metadata_base.parquet.gz
```


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = f"gs://{BUCKET_NAME}/{GCS_RAW_PREFIX}/nft_metadata_base.parquet.gz"
table_id = f"{BQ_DATASET}.nft_metadata_base"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()
print(f"Loaded: {table_id}")


## 5. Load `nft_trading_base` from parquet parts

CLI equivalent:

```bash
bq load   --source_format=PARQUET   --replace   <YOUR_GCP_PROJECT_ID>:<YOUR_BIGQUERY_DATASET>.nft_trading_base   'gs://<YOUR_GCS_BUCKET>/nft_market_regimes/raw/nft_trading_base/*.parquet'
```


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = f"gs://{BUCKET_NAME}/{GCS_RAW_PREFIX}/nft_trading_base/*.parquet"  # update if needed
table_id = f"{BQ_DATASET}.nft_trading_base"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()
print(f"Loaded: {table_id}")


## 6. Load `usd_eth_raw` from CSV

The raw Etherscan CSV is first loaded into `usd_eth_raw`.

CLI equivalent:

```bash
bq load   --source_format=CSV   --skip_leading_rows=1   --autodetect   --replace   <YOUR_GCP_PROJECT_ID>:<YOUR_BIGQUERY_DATASET>.usd_eth_raw   gs://<YOUR_GCS_BUCKET>/nft_market_regimes/raw/etherprice.csv
```


In [ ]:
csv_preview = pd.read_csv(LOCAL_RAW_DIR / 'etherprice.csv')
csv_preview.head()


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = f"gs://{BUCKET_NAME}/{GCS_RAW_PREFIX}/etherprice.csv"
table_id = f"{BQ_DATASET}.usd_eth_raw"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()
print(f"Loaded: {table_id}")


## 7. Create `usd_eth_base`

Normalize the raw Etherscan CSV into a reusable timestamp/date-based reference table.


In [ ]:
sql_usd_eth_base = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.usd_eth_base` AS
SELECT
  TIMESTAMP_SECONDS(CAST(UnixTimeStamp AS INT64)) AS timestamp,
  DATE(TIMESTAMP_SECONDS(CAST(UnixTimeStamp AS INT64))) AS date,
  DATE_TRUNC(DATE(TIMESTAMP_SECONDS(CAST(UnixTimeStamp AS INT64))), WEEK(MONDAY)) AS week_start,
  CAST(Value AS FLOAT64) AS usd_eth_rate
FROM `{BQ_DATASET}.usd_eth_raw`
WHERE TIMESTAMP_SECONDS(CAST(UnixTimeStamp AS INT64)) >= TIMESTAMP('{ANALYSIS_START_DATE}', 'UTC')
  AND TIMESTAMP_SECONDS(CAST(UnixTimeStamp AS INT64)) < TIMESTAMP('{ANALYSIS_END_DATE}', 'UTC')
ORDER BY timestamp
"""

query_job = client.query(sql_usd_eth_base)
query_job.result()
print('Created usd_eth_base')


## 8. Inspect schemas


In [ ]:
for table_name in ['nft_trading_base', 'nft_metadata_base', 'usd_eth_raw', 'usd_eth_base']:
    table = client.get_table(f"{BQ_DATASET}.{table_name}")
    print(f"\nSchema for {table_name}")
    for field in table.schema:
        print(f"- {field.name}: {field.field_type}")


## 9. Create `nft_trading_usd`

This step joins `nft_trading_base` to `usd_eth_base` on `DATE(timestamp) = date`.


In [ ]:
sql_nft_trading_usd = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.nft_trading_usd` AS
SELECT
  t.*,
  fx.usd_eth_rate,
  t.price_eth * fx.usd_eth_rate AS price_usd,
  t.fee_eth * fx.usd_eth_rate AS fee_usd
FROM `{BQ_DATASET}.nft_trading_base` AS t
LEFT JOIN `{BQ_DATASET}.usd_eth_base` AS fx
  ON DATE(t.timestamp) = fx.date
"""

query_job = client.query(sql_nft_trading_usd)
query_job.result()
print('Created nft_trading_usd')


## 10. Create `nft_trading_usd_prefilter`

This step removes collection-level and global USD-price outliers.


In [ ]:
sql_prefilter = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.nft_trading_usd_prefilter` AS
WITH
  base AS (
    SELECT
      *
    FROM `{BQ_DATASET}.nft_trading_usd`
  ),
  local_cutoff AS (
    SELECT
      collection,
      APPROX_QUANTILES(price_usd, {LOCAL_PRICE_QUANTILE_RESOLUTION})[OFFSET({LOCAL_PRICE_QUANTILE_OFFSET})] AS local_p999
    FROM base
    WHERE price_usd IS NOT NULL
    GROUP BY collection
  )
SELECT
  b.*
FROM base b
LEFT JOIN local_cutoff lc USING (collection)
WHERE
  (lc.local_p999 IS NULL OR b.price_usd <= lc.local_p999)
  AND b.price_usd <= {GLOBAL_MAX_PRICE_USD}
"""

query_job = client.query(sql_prefilter)
query_job.result()
print('Created nft_trading_usd_prefilter')


## 11. Create `nft_trading_usd_filtered`

This step retains only collections observed in at least 8 active weeks.


In [ ]:
sql_filtered = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.nft_trading_usd_filtered` AS
WITH base AS (
  SELECT *
  FROM `{BQ_DATASET}.nft_trading_usd_prefilter`
),
collection_stats AS (
  SELECT
    collection,
    COUNT(DISTINCT week_start) AS active_weeks
  FROM base
  GROUP BY collection
),
active_collections AS (
  SELECT collection
  FROM collection_stats
  WHERE active_weeks >= {MIN_ACTIVE_WEEKS}
),
final_trades AS (
  SELECT b.*
  FROM base b
  JOIN active_collections USING(collection)
)
SELECT *
FROM final_trades
"""

query_job = client.query(sql_filtered)
query_job.result()
print('Created nft_trading_usd_filtered')


## 12. Validation queries


In [ ]:
validation_sql = f"""
SELECT 'nft_trading_base' AS table_name, COUNT(*) AS n FROM `{BQ_DATASET}.nft_trading_base`
UNION ALL
SELECT 'nft_metadata_base', COUNT(*) FROM `{BQ_DATASET}.nft_metadata_base`
UNION ALL
SELECT 'usd_eth_raw', COUNT(*) FROM `{BQ_DATASET}.usd_eth_raw`
UNION ALL
SELECT 'usd_eth_base', COUNT(*) FROM `{BQ_DATASET}.usd_eth_base`
UNION ALL
SELECT 'nft_trading_usd', COUNT(*) FROM `{BQ_DATASET}.nft_trading_usd`
UNION ALL
SELECT 'nft_trading_usd_prefilter', COUNT(*) FROM `{BQ_DATASET}.nft_trading_usd_prefilter`
UNION ALL
SELECT 'nft_trading_usd_filtered', COUNT(*) FROM `{BQ_DATASET}.nft_trading_usd_filtered`
"""

client.query(validation_sql).to_dataframe()
